# Imputation experiments (SQ1/SQ2)


## 1. Path setup + dataset

In [ ]:
# ---------------------------------------------------------------------------
# Path setup + dataset switch. This is the ONLY cell you change between
# datasets: set DATASET to a key registered in common.datasets.
# ---------------------------------------------------------------------------
import sys
from pathlib import Path

DATASET = "scenario33"          # "scenario5" | "scenario33"


def _find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "src").is_dir() and (parent / "data").is_dir():
            return parent
    return start.parent


REPO_ROOT   = _find_repo_root(Path.cwd())
SRC         = REPO_ROOT / "src"
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
CSV_DIR     = RESULTS_DIR / "csv"
TUNED_PARAMS_PATH = RESULTS_DIR / "tuned_params.json"
for _d in (FIGURES_DIR, CSV_DIR):
    _d.mkdir(parents=True, exist_ok=True)

_diffputer = REPO_ROOT / "external" / "DiffPuter"
for _p in (REPO_ROOT, SRC,
           _diffputer / "baselines" / "GRAPE",
           _diffputer / "baselines",
           _diffputer):
    _sp = str(_p.resolve())
    if _p.exists() and _sp not in sys.path:
        sys.path.insert(0, _sp)

from common.datasets import get_dataset

spec = get_dataset(DATASET)
DATA_DIR = REPO_ROOT / "data" / spec.data_dirname
print(f"Dataset: {spec.name}")

print("DATA_DIR:", DATA_DIR)

In [ ]:
# Check environment and imports
from common.environment import check_environment
check_environment()
# Enable automatic reloading of edited modules in this notebook
%load_ext autoreload
%autoreload 2

## 2. Dataset analysis (raw)

In [ ]:
import pandas as pd
import numpy as np

# Raw CSV exploration (pre-cleaning), purely informational.
df = pd.read_csv(DATA_DIR / spec.raw_csv_name)
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

## 3. Load clean data

Parsing + column drops are handled by the dataset's loader.

In [ ]:
clean_data = spec.loader(DATA_DIR)
clean_data.head()

# Amputation experiments: MCAR, MAR, and MNAR

In [ ]:
from imputers.imputers import (
    MeanImputer, KNNImputerWrapper, MICEImputer,
    SoftImputeWrapper, HyperImputeImputer,
)
from imputers.diffputer_imputer import DiffPuterImputer
from imputers.grape_imputer import GRAPEImputer

from common.experiment import ExperimentConfig, run_experiments

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — built from the dataset spec.
# ---------------------------------------------------------------------------
cfg = ExperimentConfig(
    target_cols=spec.target_cols,
    retained_cols=spec.retained_cols,
    mar_driver_cols=spec.mar_driver_cols,
    mar_score_mode=spec.mar_score_mode,
    proportions=[0.10, 0.30, 0.50],
    n_seeds=3,
)

# Run experiments

In [ ]:
from common.tuning import load_tuned_params

# Load the SAME tuned params used downstream so the two tables stay comparable.
# Index keys directly and assert presence: a missing tuned value should fail
# loudly, not silently fall back to a library default.
tp = load_tuned_params(TUNED_PARAMS_PATH, spec.name)
assert tp, (f"No tuned params for '{spec.name}' in {TUNED_PARAMS_PATH}. "
            f"Run tuning.ipynb with DATASET='{spec.name}' first.")
for _k in ("knn_n_neighbors", "softimpute_shrinkage", "diffputer",
           "grape_epochs", "grape_node_edge_dim", "grape_lr"):
    assert _k in tp, f"tuned_params.json missing key '{_k}' for {spec.name}."
print("Loaded tuned params:",
      {k: ("{...}" if isinstance(v, dict) else v) for k, v in tp.items()})

mean        = MeanImputer()
knn         = KNNImputerWrapper(k=tp["knn_n_neighbors"])
mice        = MICEImputer()
softimpute  = SoftImputeWrapper(shrinkage_value=tp["softimpute_shrinkage"])
hyperimpute = HyperImputeImputer()
diffputer   = DiffPuterImputer(**tp["diffputer"])
grape       = GRAPEImputer(epochs=tp["grape_epochs"],
                           node_dim=tp["grape_node_edge_dim"],
                           edge_dim=tp["grape_node_edge_dim"],
                           lr=tp["grape_lr"])

imputers = [mean, knn, mice, softimpute, hyperimpute, diffputer, grape]

In [ ]:
scenario_filter = ["MCAR", "MAR", "MNAR", "MCAR-Row"]
results_df, summary_df = run_experiments(
    clean_data, config=cfg, imputers=imputers,
    scenario_filter=scenario_filter, verbose=False,
)
display(summary_df)

## Visualise + persist

In [ ]:
from common.visualization import plot_metric_bars, err_table, stat_fidelity_table

RECON_CSV = CSV_DIR / f"{spec.name}_imputation_results.csv"

# Persist this run, then read back from CSV so plots/tables are reproducible
# from disk (uncomment the first time, or whenever you re-run the sweep).
# results_df.to_csv(RECON_CSV, index=False)
data_source = RECON_CSV if RECON_CSV.exists() else results_df

scenario_filter = ["MCAR", "MAR", "MNAR", "MCAR-Row"]
plot_metric_bars(data_source, "rmse", "RMSE", scenarios=["MCAR", "MAR", "MNAR"],
                 filename=FIGURES_DIR / f"{spec.name}_rmse.png")

for mech in ["MCAR", "MAR", "MNAR"]:
    display(err_table(data_source, mech))

display(stat_fidelity_table(data_source, scenario="MCAR", proportion=0.3))

## Row-wise MCAR (separate run, appended)

In [ ]:
# # Row-wise MCAR only — separate run, same imputers/config.
# cfg.scenarios = cfg.scenarios + [("MCAR-Row", "mcar_rowwise")]
# row_results, _ = run_experiments(
#     clean_data, config=cfg, imputers=imputers,
#     scenario_filter=["MCAR-Row"], verbose=True,
# )

# main_csv = CSV_DIR / f"{spec.name}_imputation_results.csv"
# if main_csv.exists():
#     existing = pd.read_csv(main_csv)
#     combined = (pd.concat([existing, row_results], ignore_index=True)
#                   .drop_duplicates(subset=["scenario", "proportion", "method", "seed"],
#                                    keep="last"))
# else:
#     combined = row_results
# combined.to_csv(main_csv, index=False)
# print(f"Appended {len(row_results)} rows; total now {len(combined)} in {main_csv.name}.")